In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from sklearn.metrics import r2_score

df = pd.read_csv('site_params.csv')

x = df['SQRT(D84/D16)'].values
a_vals = df['A'].values
b_vals = df['B'].values
ks_vals = df['Ks'].values
kw_vals = df['Kw'].values
sites = df['site'].values

In [4]:
# determine colors for each site
site_colors = {
    'erlenbach': '#f28e2b',   # orange
    'flume': '#1f77b4',       # blue
    'la jara': '#2ca02c'      # green
}

colors = [site_colors[s.lower()] for s in sites]

# fit functions
def power_law(x, A, b):
    return A * (x ** b)

def linear(x, m, c):
    return m * x + c

# fit power law to A vs sqrt(D84/D16)
a_params, _ = curve_fit(power_law, x, a_vals)
a_A_fit, a_b_fit = a_params
x_line = np.linspace(min(x), max(x), 300)
y_line = power_law(x_line, a_A_fit, a_b_fit)
# R²
a_pred = power_law(x, a_A_fit, a_b_fit)
r2 = r2_score(a_vals, a_pred)

# fit power law to B vs sqrt(D84/D16)
b_params, _ = curve_fit(power_law, x, b_vals)
b_A_fit, b_b_fit = b_params
y_line_b = power_law(x_line, b_A_fit, b_b_fit)
# R²
b_pred = power_law(x, b_A_fit, b_b_fit)
r2_b = r2_score(b_vals, b_pred)

# fit linear to Ks vs sqrt(D84/D16)
ks_m, ks_c = np.polyfit(x, ks_vals, 1)
ks_y_line = linear(x_line, ks_m, ks_c)
pred = linear(x, ks_m, ks_c)
r2 = r2_score(ks_vals, pred)

# fit linear to Kw vs sqrt(D84/D16)
kw_m, kw_c = np.polyfit(x, kw_vals, 1)
kw_y_line = linear(x_line, kw_m, kw_c)
pred = linear(x, kw_m, kw_c)   
r2_kw = r2_score(kw_vals, pred)

# print results
print(f"A fit: A={a_A_fit:.3f}, b={a_b_fit:.3f}, R²={r2:.3f}")
print(f"B fit: A={b_A_fit:.3f}, b={b_b_fit:.3f}, R²={r2_b:.3f}")
print(f"Ks fit: m={ks_m:.3f}, c={ks_c:.3f}, R²={r2:.3f}")
print(f"Kw fit: m={kw_m:.3f}, c={kw_c:.3f}, R²={r2_kw:.3f}")

RuntimeError: Optimal parameters not found: Number of calls to function has reached maxfev = 600.

In [8]:
def fit_powerlaw_excel(x, y):
    # remove zeros or negatives
    mask = (x > 0) & (y > 0)
    x_fit = x[mask]
    y_fit = y[mask]
    # log transform
    logx = np.log(x_fit)
    logy = np.log(y_fit)
    # linear regression in log space
    b, logA = np.polyfit(logx, logy, 1)
    A = np.exp(logA)
    # predictions
    y_pred = A * (x_fit ** b)
    r2 = r2_score(y_fit, y_pred)

    return A, b, r2


# using the excel approach 
a_A_fit, a_b_fit, r2_a = fit_powerlaw_excel(x, a_vals)
x_line = np.linspace(min(x), max(x), 300)
y_line_a = a_A_fit * (x_line ** a_b_fit)

b_A_fit, b_b_fit, r2_b = fit_powerlaw_excel(x, b_vals)
y_line_b = b_A_fit * (x_line ** b_b_fit)

ks_m, ks_c = np.polyfit(x, ks_vals, 1)
ks_y_line = linear(x_line, ks_m, ks_c)
ks_pred = linear(x, ks_m, ks_c)
r2_ks = r2_score(ks_vals, ks_pred)

kw_m, kw_c = np.polyfit(x, kw_vals, 1)
kw_y_line = linear(x_line, kw_m, kw_c)
kw_pred = linear(x, kw_m, kw_c)
r2_kw = r2_score(kw_vals, kw_pred)

print(f"A fit: A={a_A_fit:.3f}, b={a_b_fit:.3f}, R²={r2_a:.3f}")
print(f"B fit: A={b_A_fit:.3f}, b={b_b_fit:.3f}, R²={r2_b:.3f}")
print(f"Ks fit: m={ks_m:.3f}, c={ks_c:.3f}, R²={r2_ks:.3f}")
print(f"Kw fit: m={kw_m:.3f}, c={kw_c:.3f}, R²={r2_kw:.3f}")

A fit: A=0.063, b=-3.975, R²=-0.414
B fit: A=11.770, b=-0.944, R²=0.786
Ks fit: m=-24.469, c=93.595, R²=0.970
Kw fit: m=-2.392, c=10.343, R²=0.386
